# Energy Consumption Forecasting via Recurrent Neural Network Architectures
### A Comparative Study of LSTM, GRU, BiLSTM, and BiGRU Models
>**Author:** Erdem YILMAZ  
**Institution:** Adana Alparslan Türkeş Science and Technology University  
**Department:** Artificial Intelligence Engineering  
**Date:** June 2026
**GitHub:** https://github.com/eerdemYlmzz
 **LinkedIn:** www.linkedin.com/in/erdem-yilmaz-24818727a









In [21]:
# import torch
# import torch.nn as nn
# import pandas as pd
# import numpy as np
# import math
# import random
# import time
# import matplotlib.pyplot as plt

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# df=pd.read_csv("/content/drive/MyDrive/powerconsumption.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# df.head(20)

,Datetime,Temperature,Humidity,WindSpeed,GeneralDiffuseFlows,DiffuseFlows,PowerConsumption_Zone1,PowerConsumption_Zone2,PowerConsumption_Zone3
0,1/1/2017 0:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
1,1/1/2017 0:10,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2,1/1/2017 0:20,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
3,1/1/2017 0:30,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
4,1/1/2017 0:40,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964
5,1/1/2017 0:50,5.853,76.9,0.081,0.059,0.108,26624.81013,17416.41337,18130.12048
6,1/1/2017 1:00,5.641,77.7,0.080,0.048,0.096,25998.98734,16993.31307,17945.06024
7,1/1/2017 1:10,5.496,78.2,0.085,0.055,0.093,25446.07595,16661.39818,17459.27711
8,1/1/2017 1:20,5.678,78.1,0.081,0.066,0.141,24777.72152,16227.35562,17025.54217
9,1/1/2017 1:30,5.491,77.3,0.082,0.062,0.111,24279.49367,15939.20973,16794.21687


In [ ]:
# df.rename(columns={"Datetime":"time","PowerConsumption_Zone1":"consumption"},inplace=True)

In [ ]:
# df=df.drop(columns=["Temperature",	"Humidity",	"WindSpeed",	"GeneralDiffuseFlows",	"DiffuseFlows","PowerConsumption_Zone2",	"PowerConsumption_Zone3"])

In [ ]:
# df.head()

,time,consumption
0,1/1/2017 0:00,34055.69620
1,1/1/2017 0:10,29814.68354
2,1/1/2017 0:20,29128.10127
3,1/1/2017 0:30,28228.86076
4,1/1/2017 0:40,27335.69620


In [ ]:
# device ="cuda" if torch.cuda.is_available() else "cpu"
# device

'cuda'

In [ ]:
# x,y=df.shape
# x

52416

In [ ]:
# df["time"]=df["time"].apply(lambda x:pd.to_datetime(x).strftime("%H:%M"))

# df.to_parquet("/content/drive/MyDrive/processed_powerconsumption.parquet", index=False)

**Mounted data. **EXECUTE** the cells from this point**








---



In [23]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import math
import random
import time
import matplotlib.pyplot as plt

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/processed_powerconsumption.parquet")

print(df.head())

    time  consumption
0  00:00    34.055696
1  00:10    29.814684
2  00:20    29.128101
3  00:30    28.228861
4  00:40    27.335696


In [26]:
df["consumption"]=df["consumption"] /1000

In [27]:
df["consumption"]

,consumption
0,0.034056
1,0.029815
2,0.029128
3,0.028229
4,0.027336
...,...
52411,0.031160
52412,0.030430
52413,0.029591
52414,0.028958


In [28]:
df

,time,consumption
0,00:00,0.034056
1,00:10,0.029815
2,00:20,0.029128
3,00:30,0.028229
4,00:40,0.027336
...,...,...
52411,23:10,0.031160
52412,23:20,0.030430
52413,23:30,0.029591
52414,23:40,0.028958


In [29]:
# plt.plot(df["time"],df["consumption"])

In [30]:
#plt.bar(df["time"],df["consumption"],color="skyblue")

In [31]:
len(df)

52416

In [32]:
x,y=df.shape

x / 144

day=[]
for i in range(0,len(df),144):
  each_day=df.iloc[i:i+144]
  day.append(each_day)




##**Annual electricity consumption records**.





In [33]:
day_list = [pd.DataFrame(i, columns=['time', 'consumption']) for i in day]
df_day = pd.concat(day_list, ignore_index=True)

In [34]:
df_day

,time,consumption
0,00:00,0.034056
1,00:10,0.029815
2,00:20,0.029128
3,00:30,0.028229
4,00:40,0.027336
...,...,...
52411,23:10,0.031160
52412,23:20,0.030430
52413,23:30,0.029591
52414,23:40,0.028958


In [35]:
type(df_day) # means day 3

pandas.core.frame.DataFrame

In [36]:
df_day.index

RangeIndex(start=0, stop=52416, step=1)

In [37]:
n=144
daily_sum=df_day["consumption"].groupby(df_day.index//n).sum()
daily_sum
df_daily_sum=pd.DataFrame({
    'day': [f'Day {i+1}' for i in range(len(daily_sum))],
    'total_consumption': daily_sum.values
})
df_daily_sum

,day,total_consumption
0,Day 1,4.098993
1,Day 2,4.157207
2,Day 3,4.400992
3,Day 4,4.419336
4,Day 5,4.435619
...,...,...
359,Day 360,4.321941
360,Day 361,4.315243
361,Day 362,4.358449
362,Day 363,4.206187


In [38]:
daily=df_daily_sum

In [39]:
daily

,day,total_consumption
0,Day 1,4.098993
1,Day 2,4.157207
2,Day 3,4.400992
3,Day 4,4.419336
4,Day 5,4.435619
...,...,...
359,Day 360,4.321941
360,Day 361,4.315243
361,Day 362,4.358449
362,Day 363,4.206187


In [40]:
from sklearn.preprocessing import MinMaxScaler

# Correctly select the 'total_consumption'  column for scaling
consumption_values = daily['total_consumption'].values.reshape(-1, 1)

scaler=MinMaxScaler(feature_range=(-1,1))
scaled_data=scaler.fit_transform(consumption_values)
scaled_data

array([[-7.16817201e-01],
       [-6.49226100e-01],
       [-3.66170286e-01],
       [-3.44872140e-01],
       [-3.25965539e-01],
       [-3.41126094e-01],
       [-3.57768136e-01],
       [-5.33790009e-01],
       [-3.69881060e-01],
       [-1.48673819e-01],
       [-2.83355138e-01],
       [-1.13118120e-01],
       [-2.13414821e-01],
       [-3.20653348e-01],
       [-1.00000000e+00],
       [-3.62022121e-01],
       [-1.53964846e-01],
       [ 1.80570111e-02],
       [-1.02373809e-01],
       [-1.62070698e-01],
       [-1.37851906e-01],
       [-3.06106552e-01],
       [-4.13153621e-02],
       [-1.95495877e-01],
       [-7.50438929e-02],
       [-9.67502286e-03],
       [-7.62008642e-02],
       [-2.43150390e-01],
       [-5.83836066e-01],
       [-2.05400679e-01],
       [-1.93703983e-01],
       [-2.30448967e-01],
       [-1.45724296e-01],
       [-3.37150051e-01],
       [-3.48754618e-01],
       [-7.51124188e-01],
       [-3.69817119e-01],
       [-3.41188271e-01],
       [-3.5

In [41]:
def create_dataset(scaled_data, lookback=7):
    X, y = [], []
    for i in range(len(scaled_data) - lookback):
        X.append(scaled_data[i:(i + lookback), 0])
        y.append(scaled_data[i + lookback, 0])
    return np.array(X), np.array(y)


In [42]:
lookback=7
X,y=create_dataset(scaled_data,lookback)

In [43]:
y.shape

(357,)

In [44]:
X = X.reshape(X.shape[0], X.shape[1], 1)
y=y.reshape(y.shape[0],1)

In [45]:
split_index= int(len(X)*0.95)
split_index

339

In [46]:
X_train=X[:split_index]
X_test=X[split_index:]
y_test=y[split_index:]
y_train=y[:split_index]

In [47]:
print(X_train.shape)
print(type(X))

(339, 7, 1)
<class 'numpy.ndarray'>


In [48]:
X_train=torch.tensor(X_train).float()
y_train=torch.tensor(y_train).float()
X_test=torch.tensor(X_test).float()
y_test=torch.tensor(y_test).float()
y_test.shape
X_test.shape

torch.Size([18, 7, 1])

In [49]:
from torch.utils.data import Dataset
class TimeSeriesDataset(Dataset):
  def __init__(self,X,y):
    self.X=X
    self.y=y

  def __len__(self):
    return len(self.X)
  def __getitem__(self,i):
    return self.X[i],self.y[i]

train_data=TimeSeriesDataset(X_train,y_train)
test_data=TimeSeriesDataset(X_test,y_test)


In [50]:
from torch.utils.data import DataLoader
batch_size=16
train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(test_data,batch_size=batch_size,shuffle=False)

In [51]:
class LSTMForecaster(nn.Module):
  def __init__(self):
    super().__init__()
    self.lstm=nn.LSTM(input_size=1,hidden_size=32,num_layers=1,bias=True,batch_first=True)
    self.linear=nn.Linear(in_features=32,out_features=1)
    self.dropout=nn.Dropout(p=0.5)

  def forward(self,x):
    x,_= self.lstm(x)
    x = x[:, -1, :]
    x=self.dropout(x)
    x=self.linear(x)
    return x


In [52]:
import torch.optim as optim
import torch.utils.data as data

In [53]:
model=LSTMForecaster()
optimizer=optim.Adam(model.parameters(),lr=0.01)
loss_f = nn.MSELoss()


In [54]:

n_epochs=6000

for epoch in range(n_epochs):
  model.train()

  for X_batch,y_batch in train_loader:
    y_pred=model(X_batch)
    loss = loss_f(y_pred.squeeze(1), y_batch.squeeze(1))
    optimizer.zero_grad()
    loss.backward()
    gradient_clipping=nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
    optimizer.step()

  if epoch % 100 !=0:
    continue
  model.eval()
  with torch.no_grad():
    y_pred_t=model(X_train)
    train_rmse = np.sqrt(loss_f(y_pred_t.squeeze(1), y_train.squeeze(1)).item())
    y_pred_test=model(X_test)
    test_rmse = np.sqrt(loss_f(y_pred_test.squeeze(1), y_test.squeeze(1)).item())
  print("Epoch %d: train RMSE %.4f,test RMSE %.4f" % (epoch,train_rmse,test_rmse))


Epoch 0: train RMSE 0.2498,test RMSE 0.1807
Epoch 100: train RMSE 0.1996,test RMSE 0.2026
Epoch 200: train RMSE 0.1693,test RMSE 0.1112
Epoch 300: train RMSE 0.1490,test RMSE 0.1248
Epoch 400: train RMSE 0.1352,test RMSE 0.0909
Epoch 500: train RMSE 0.0994,test RMSE 0.0905
Epoch 600: train RMSE 0.0919,test RMSE 0.0915
Epoch 700: train RMSE 0.0767,test RMSE 0.1071
Epoch 800: train RMSE 0.0882,test RMSE 0.1081
Epoch 900: train RMSE 0.0747,test RMSE 0.0711
Epoch 1000: train RMSE 0.0734,test RMSE 0.0800
Epoch 1100: train RMSE 0.0605,test RMSE 0.1148
Epoch 1200: train RMSE 0.0650,test RMSE 0.0689
Epoch 1300: train RMSE 0.0642,test RMSE 0.0900
Epoch 1400: train RMSE 0.0588,test RMSE 0.0781
Epoch 1500: train RMSE 0.0613,test RMSE 0.0756
Epoch 1600: train RMSE 0.0533,test RMSE 0.1004
Epoch 1700: train RMSE 0.0443,test RMSE 0.0836
Epoch 1800: train RMSE 0.0621,test RMSE 0.0803
Epoch 1900: train RMSE 0.0474,test RMSE 0.1136
Epoch 2000: train RMSE 0.0431,test RMSE 0.0905
Epoch 2100: train RMSE 0.

KeyboardInterrupt: 


try with higher num_layer and  more than 1 LSTM layer.